In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.model_selection import train_test_split

def remove_duplicates(text):
    # Remove special characters and split the text into words
    words = re.findall(r'\b\w+\b', text)
    unique_words = []
    for word in words:
        if word not in unique_words:
            unique_words.append(word.lower())
    return ' '.join(unique_words)

data = pd.read_json('./data/youtube_history.json')
data = data.drop(columns=['url', 'visit_count', 'last_visit_time', 'publishedAt', 'description', 'channelTitle', 'channelId'])
data['tags'] = data['tags'].fillna('').apply(lambda x: list(set(x))).apply(lambda x: ' '.join(map(str, x))).apply(lambda x: x.strip())

data['tags'] = data['tags'].apply(remove_duplicates)
data['title'] = data['title'].apply(remove_duplicates)

data['text'] = data['title'] + ' ' + data['tags']

if 'topics' not in data.columns:
	# Add a new column initialized as an empty list
	data['topics'] = [[]] * len(data)
	
if 'topic_words'	not in data.columns:
	# Add a new column initialized as an empty list
	data['topic_words'] = [[]] * len(data)

data.info()

In [ ]:
def create_topics(data, topic_count):
  vectorizer = TfidfVectorizer(stop_words='english')
  X = vectorizer.fit_transform(data['text'])
 
  nmf = NMF(n_components=topic_count, random_state=42)
  W = nmf.fit_transform(X)
  H = nmf.components_

  feature_names = vectorizer.get_feature_names_out()
  top_words_list = []
  for topic_idx, topic in enumerate(H):
    topic_words = " ".join([feature_names[i] for i in topic.argsort()[:-21:-1]])
    top_words_list.append(topic_words)
    
  topic_assignments = W.argmax(axis=1)
  top_words = [top_words_list[i] for i in topic_assignments]
  topics = [topic_assignments[i] for i in range(len(topic_assignments))]
  for index, row in data.iterrows():
    row['topics'].append(topics[index])
    row['topic_words'].append(top_words[index])
  

In [ ]:
create_topics(data, 5)

In [ ]:
data['topics']

In [ ]:
# Create a TF-IDF matrix
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(data['text'])

topic_count = 5

# Use Non-negative Matrix Factorization (NMF) to extract topics
nmf = NMF(n_components=topic_count, random_state=42)
W = nmf.fit_transform(X)
H = nmf.components_

# Get the topic assignments for each row
topic_assignments = W.argmax(axis=1)

# Count the number of rows matched to each topic
topic_counts = [sum(topic_assignments == i) for i in range(topic_count)]

data['first_topic_id'] = topic_assignments

# Print the top words for each topic
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    topic_words = " ".join([feature_names[i] for i in topic.argsort()[:-21:-1]])
    print(f"Topic[{topic_idx}] {topic_counts[topic_idx]} matches: {topic_words}")

print()
data.info()

In [ ]:
# Create a list to store the top words for each topic
top_words_list = []

# Print the items for each topic
for topic_idx, topic in enumerate(H):
    topic_words = [feature_names[i] for i in topic.argsort()[:-21:-1]]
    top_words_list.append(" ".join(topic_words))


data['top_words'] = [top_words_list[i] for i in topic_assignments]

#data.iloc[2805]['top_words']
#data.style.set_properties(**{'text-align': 'left'}).set_table_styles([ dict(selector='th', props=[('text-align', 'left')] ) ])

topic_four = data[data['first_topic_id'] == 4]
topic_four.info()